###DAY 2 (21/02/26) – Feature Table & Silver Layer
###🏗️ Architecture & Strategy
Welcome to Day 2! Today we implement the **Medallion Architecture**, specifically moving data from the **Bronze layer** (raw, historical events) to the **Silver layer** (cleaned, enriched, and structured data).
In Machine Learning, algorithms don't learn from individual "clicks" very well; they learn from "behaviors." Our goal today is to transition from an event-level table to an entity-level feature table.

**Our Strategy**:

1. **Deduplication**: Raw event logs often contain duplicate triggers due to network retries or tracking bugs. We must filter these out before calculating metrics.

2. **Feature Engineering**: We will aggregate billions of events into a clean, user-level profile. We will calculate behavioral metrics like total spend, purchase counts, and cart-to-purchase ratios.

3. **Data Quality Validation**: A model is only as good as its data. We will actively validate that our table has no duplicate users and handle null values properly.

4. **Silver Storage & Z-Ordering**: We will save this as a Managed Delta Table and use `ZORDER BY (user_id)`. Since downstream ML models and Gold tables will frequently join on `user_id`, this optimization will make those queries exponentially faster.

###Setup & Deduplication (Bronze Preparation)
Before aggregating, we load our managed table from Day 1 and ensure the foundation is perfectly clean.

In [0]:
from pyspark.sql import functions as F

# 1. Setup Context (Assuming Unity Catalog is configured)
catalog_name = "course_catalog"  
schema_name = "ecommerce_governed"
bronze_table_name = f"{catalog_name}.{schema_name}.events_delta_managed"

print(f"🔄 Setting context to: {catalog_name}.{schema_name}")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

# 2. Load the Bronze Data
print(f"⏳ Loading Bronze data from: {bronze_table_name}...")
df_bronze = spark.table(bronze_table_name)

# 3. Deduplication
# Network retries can cause duplicate events with the exact same timestamp, user, and product.
# We drop exact duplicates across all columns.
print("🧹 Cleaning data: Removing exact duplicates...")
initial_count = df_bronze.count()
df_clean = df_bronze.dropDuplicates()
clean_count = df_clean.count()

duplicates_removed = initial_count - clean_count
print(f"✅ Deduplication complete. Removed {duplicates_removed:,} duplicate rows.")

###Feature Engineering (User-Level Aggregation)
We transform the data from `one row = one event` to one `row = one user`. We create rich ML features that represent user intent and purchasing power.

In [0]:
# ---------------------------------------------------------
# FEATURE ENGINEERING: BUILDING THE USER PROFILE
# ---------------------------------------------------------
print("⚙️ Engineering user-level features...")

# Grouping by user_id to create behavioral metrics
df_user_features = df_clean.groupBy("user_id").agg(
    # Engagement Features
    F.count("*").alias("total_events"),
    F.count(F.when(F.col("event_type") == "view", 1)).alias("view_count"),
    F.count(F.when(F.col("event_type") == "cart", 1)).alias("cart_count"),
    
    # Conversion Features
    F.count(F.when(F.col("event_type") == "purchase", 1)).alias("purchase_count"),
    
    # Monetary Features (Handling nulls if they only viewed but never bought)
    F.coalesce(F.sum(F.when(F.col("event_type") == "purchase", F.col("price"))), F.lit(0.0)).alias("total_spent"),
    F.coalesce(F.avg(F.when(F.col("event_type") == "purchase", F.col("price"))), F.lit(0.0)).alias("avg_purchase_price")
)

# Adding a derived feature: Conversion Rate (Purchases per View)
# We add +1 to the denominator to prevent DivisionByZero errors for users who only have 'cart' events but no 'views'
df_user_features = df_user_features.withColumn(
    "view_to_purchase_ratio",
    F.round((F.col("purchase_count") / (F.col("view_count") + 1)), 4)
)

print("✅ Feature engineering complete.")
display(df_user_features.limit(5))

###Validate Feature Quality & Visualize Distributions
Data engineering requires defensive programming. We must prove our table is statistically sound before saving it.

In [0]:
# ---------------------------------------------------------
# DATA QUALITY VALIDATION
# ---------------------------------------------------------
print("🛡️ Running Data Quality Validation Checks...")

# Check 1: Ensure User IDs are strictly unique (Primary Key constraint)
total_users = df_user_features.count()
distinct_users = df_user_features.select("user_id").distinct().count()

if total_users == distinct_users:
    print(f"   ✅ PASS: 'user_id' is perfectly unique ({total_users:,} users).")
else:
    print(f"   ❌ FAIL: Found duplicate user_ids! Total: {total_users}, Distinct: {distinct_users}")

# Check 2: Check for missing / null User IDs
null_users = df_user_features.filter(F.col("user_id").isNull()).count()
if null_users == 0:
    print("   ✅ PASS: No null 'user_id' records found.")
else:
    print(f"   ⚠️ WARNING: Found {null_users} records with null user_id. Filtering them out...")
    df_user_features = df_user_features.filter(F.col("user_id").isNotNull())

# ---------------------------------------------------------
# VISUALIZE DATA DISTRIBUTIONS
# ---------------------------------------------------------
print("📊 Visualizing User Spend Distribution...")
# We filter to only show users who actually bought something to see the spend distribution clearly
normal_purchasers = df_user_features.filter(
    (F.col("total_spent") > 0) & (F.col("total_spent") < 5000)
).select("total_spent")

display(normal_purchasers)

# 💡 INSTRUCTIONS FOR DATABRICKS NATIVE PLOTTING:
# 1. Click the '+' icon on the display output -> 'Visualization'.
# 2. Select 'Histogram'.
# 3. X-Axis: 'total_spent'.
# 4. This will show you the skew of customer spending!

Databricks visualization. Run in Databricks to view.

###Save to Silver Layer & Optimize (Z-ORDER)
Finally, we write the feature table to the Silver layer. Because this table will be joined with other tables using `user_id`, we apply `ZORDER BY (user_id)` to massively speed up future machine learning data prep.

In [0]:
# ---------------------------------------------------------
# SAVE AND OPTIMIZE THE SILVER TABLE
# ---------------------------------------------------------
silver_table_name = f"{catalog_name}.{schema_name}.silver_user_features"

print(f"💾 Saving to Silver Layer: {silver_table_name}...")

# Write to Managed Delta Table
df_user_features.write.format("delta").mode("overwrite").saveAsTable("silver_user_features")

print("🚀 Optimizing Table and applying Z-Ordering on 'user_id'...")
# Z-Ordering physically sorts the data in the parquet files based on user_id.
# When a downstream query looks for a specific user, Delta Lake can skip 99% of the files.
optimize_metrics = spark.sql("OPTIMIZE silver_user_features ZORDER BY (user_id)")

display(optimize_metrics)
print("🏆 Day 2 Pipeline Complete!")